# First day-ahead price model experiment

Use this Databricks notebook to inspect labelled Gold data, select a consistent auction series and resolution, then compare the seasonal baseline with the global LightGBM model. Training logic lives in `eu_electricity_forecast.training`; runs are logged to MLflow and are not registered automatically.

In [ ]:
import json
import os
import sys
import pandas as pd

# Use the Python package from this repo's Databricks-synced source tree.
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
repo_root = notebook_path.split("/src/notebooks/", 1)[0]
sys.path.insert(0, os.path.join(repo_root, "src"))

from eu_electricity_forecast.training import gold_source_versions, load_training_frame, run_experiment, target_coverage

dbutils.widgets.text("catalog", "eu_electricity_forecast", "Catalog")
dbutils.widgets.text("gold_schema", "gold", "Gold schema")
dbutils.widgets.text("resolution_minutes", "15", "Delivery resolution in minutes")
dbutils.widgets.text("series_by_zone_json", "{}", "Selected auction-series keys by zone (JSON)")

CATALOG = dbutils.widgets.get("catalog")
GOLD_SCHEMA = dbutils.widgets.get("gold_schema")
TABLE = f"{CATALOG}.{GOLD_SCHEMA}.day_ahead_training_data"
EXPERIMENT_PATH = f"/Users/{spark.sql('SELECT current_user()').first()[0]}/eu-electricity-forecast-day-ahead-price"
print(f"Training source: {TABLE}")
print(f"MLflow experiment: {EXPERIMENT_PATH}")

## Inspect target coverage

Choose series keys per zone that represent the same auction definition. The key includes zone and is therefore different for each zone. Do not mix the two DE_LU price series. Train one delivery resolution at a time. Historical backfill may leave lag features unavailable; inspect the lag counts before interpreting model performance.

In [ ]:
coverage = target_coverage(spark, TABLE)
display(coverage)

## Run a controlled experiment

After reviewing the table above, set `series_by_zone_json` to a JSON object mapping all four zones (`DE_LU`, `FR`, `NL`, `LT`) to their `target_auction_series_key`, and set one resolution. Example shape: `{'DE_LU': '<key>', 'FR': '<key>', 'NL': '<key>', 'LT': '<key>'}`. Select the same auction definition across zones.

In [ ]:
SERIES_BY_ZONE = json.loads(dbutils.widgets.get("series_by_zone_json"))
RESOLUTION_MINUTES = int(dbutils.widgets.get("resolution_minutes"))
if not SERIES_BY_ZONE:
    print("Select target series keys in the notebook widgets, then rerun this cell.")
else:
    source_versions = gold_source_versions(spark, TABLE)
    training_frame = load_training_frame(spark, TABLE, SERIES_BY_ZONE, RESOLUTION_MINUTES, source_versions)
    print(f"Selected labelled rows: {len(training_frame):,}")
    print(f"Zones: {sorted(training_frame['bidding_zone'].unique())}")
    print(f"Price-lag availability: {training_frame[['price_lag_24h_eur_mwh_available', 'price_lag_168h_eur_mwh_available']].mean().to_dict()}")
    result = run_experiment(
        training_frame,
        series_by_zone=SERIES_BY_ZONE,
        resolution_minutes=RESOLUTION_MINUTES,
        experiment_path=EXPERIMENT_PATH,
        source_table=TABLE,
        source_delta_versions=source_versions,
    )
    print(f"MLflow run: {result['run_id']}")

In [ ]:
if SERIES_BY_ZONE:
    metric_rows = []
    for split_name, scores in result["metrics"].items():
        for model_name in ("model", "seasonal_baseline"):
            metric_rows.append({"split": split_name, "model": model_name, "rows": scores["rows"], **scores[model_name]})
    display(pd.DataFrame(metric_rows))
    subgroup_rows = []
    for split_name, scores in result["metrics"].items():
        for dimension in ("by_zone", "by_month", "by_price_regime"):
            for group_value, group_scores in scores[dimension].items():
                for model_name in ("model", "seasonal_baseline"):
                    subgroup_rows.append({"split": split_name, "dimension": dimension, "group": group_value, "model": model_name, "rows": group_scores["rows"], **group_scores[model_name]})
    display(pd.DataFrame(subgroup_rows))
    display(pd.DataFrame([result["feature_importance"]]).T.rename(columns={0: "importance"}).sort_values("importance", ascending=False))

In [ ]:
if SERIES_BY_ZONE:
    test_predictions = result["predictions"]["test"]
    zone = sorted(test_predictions["bidding_zone"].unique())[0]
    points_per_day = 24 * 60 // RESOLUTION_MINUTES
    chart = test_predictions[test_predictions["bidding_zone"] == zone].sort_values("target_interval_start_utc").tail(points_per_day * 14)
    display(zone)
    ax = chart.set_index("target_interval_start_utc")[["label_price_eur_mwh", "model_prediction", "seasonal_baseline_prediction"]].plot(figsize=(14, 5), title=f"Held-out prices and predictions: {zone}")
    ax.set_ylabel("EUR/MWh")
    display(ax.figure)